# Train an aquaculture detector for AquaCulture.Net / Melatih detektor budidaya untuk AquaCulture.Net

*Made by Gravicode Studios, led by Kang Fadhil · Dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil*

This notebook trains a YOLO detector on **your own labelled photos** (shrimp, fish, or disease signs such as white spots, black gill and red
body), exports it to ONNX in the layout `AquaCulture.ComputerVision.OnnxDetector` expects, checks it, and optionally publishes it to the
Hugging Face Hub so that `ModelHub.LoadDetectorAsync` can download it.

Notebook ini melatih detektor YOLO dengan **foto berlabel milik Anda sendiri**, mengekspornya ke ONNX, mengujinya, lalu (opsional)
mengunggahnya ke Hugging Face Hub.

**Licences / lisensi.** Ultralytics (code and pretrained weights) is AGPL-3.0, and so are the weights you fine-tune with it. If your farm
software is not open source, get an Ultralytics Enterprise licence, or train a permissively licensed architecture instead (for example
RT-DETR from the `transformers` library under Apache-2.0). Only use photos you own or are allowed to use.

## 1. Dataset layout (YOLO format) / Struktur dataset

```
dataset/
  images/train/*.jpg   images/val/*.jpg
  labels/train/*.txt   labels/val/*.txt      # one line per object: class cx cy w h (normalised 0–1)
  data.yaml
```

Label tools: CVAT, Label Studio and Roboflow all export YOLO format. Practical minimum: 300–500 images per class, taken under your farm's
light, turbidity and camera, with at least 15 % kept for validation. Classes used by the AquaCulture.Net demos: `shrimp`, `fish`,
`white_spot`, `black_gill`, `red_body`, `lesion`.

In [ ]:
%pip install -q ultralytics onnx onnxruntime huggingface_hub

In [ ]:
from pathlib import Path
DATA = Path("dataset")                      # change to your dataset folder
CLASSES = ["shrimp", "white_spot", "black_gill", "red_body"]
(DATA / "data.yaml").write_text(
    f"path: {DATA.resolve()}\ntrain: images/train\nval: images/val\nnames: {dict(enumerate(CLASSES))}\n")
print((DATA / "data.yaml").read_text())

## 2. Train / Latih

In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11n.pt")                  # n = fastest for edge gateways; try s/m for more accuracy
results = model.train(data=str(DATA / "data.yaml"), imgsz=640, epochs=100, patience=20, batch=16, seed=1,
                      hsv_v=0.5, degrees=10, fliplr=0.5)   # brightness/rotation augmentation for pond water and phone photos
metrics = model.val()
print("mAP50:", metrics.box.map50, "mAP50-95:", metrics.box.map)

## 3. Export to ONNX / Ekspor ke ONNX

`OnnxDetector` expects a single input `[1, 3, 640, 640]` (RGB, 0–1, letterboxed) and the raw YOLOv8/11 output `[1, 4 + classes, 8400]`.
NMS runs in C# (`YoloDecoder`), so export **without** NMS.

In [ ]:
onnx_path = model.export(format="onnx", imgsz=640, opset=17, simplify=True, nms=False, dynamic=False)
import onnxruntime as ort, numpy as np
s = ort.InferenceSession(onnx_path)
print([ (i.name, i.shape) for i in s.get_inputs() ], [ (o.name, o.shape) for o in s.get_outputs() ])
out = s.run(None, {s.get_inputs()[0].name: np.zeros((1, 3, 640, 640), np.float32)})[0]
assert out.shape[1] == 4 + len(CLASSES), out.shape

## 4. Publish to the Hub (optional) / Unggah ke Hub (opsional)

In [ ]:
from huggingface_hub import HfApi, create_repo
REPO = "your-org/shrimp-disease-yolo11n"    # private=True keeps farm data and models in-house
create_repo(REPO, private=True, exist_ok=True)
api = HfApi()
api.upload_file(path_or_fileobj=onnx_path, path_in_repo="best.onnx", repo_id=REPO)
api.upload_file(path_or_fileobj=("---\nlicense: agpl-3.0\npipeline_tag: object-detection\n---\n"
                                 f"# {REPO}\nClasses: {', '.join(CLASSES)}\n").encode(), path_in_repo="README.md", repo_id=REPO)

## 5. Use it from .NET / Pakai dari .NET

```csharp
var hub = new ModelHub();                                          // HF_TOKEN for private repositories
using var detector = await hub.LoadDetectorAsync("your-org/shrimp-disease-yolo11n", "best.onnx",
    ["shrimp", "white_spot", "black_gill", "red_body"], acceptLicense: true);   // AGPL weights: accept consciously
var detections = detector.Detect(image);
```

Validate on photos from a farm that is **not** in the training set before you trust a disease call. Always confirm suspected WSSV or
AHPND with a PCR laboratory.